In [21]:
import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path
from collections import Counter
import random
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login
import numpy as np
import cv2

from facenet_pytorch import MTCNN

In [19]:
mtcnn = MTCNN(
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=device,
    keep_all=True
)


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,
)
pipe = pipe.to(device)
pipe.safety_checker = None  # optional; depends on your environment/policy

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 439.50it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /home/sera/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 430.95it/s, Materializing param=visual_projection.weight]
StableDiffusionSafetyChecker LOAD REPORT from: /home/sera/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+--

In [3]:
train_dir = Path("../data/dataset_processed_split/train")
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])

counts = {c: len(list((train_dir/c).glob("*.png"))) for c in classes}
max_count = max(counts.values())
target = int(0.7 * max_count)

need = {c: max(0, target - counts[c]) for c in classes}
need

{'angry': 812,
 'disgusted': 3475,
 'fearful': 695,
 'happy': 0,
 'neutral': 75,
 'sad': 153,
 'surprised': 1402}

In [4]:
ages = ["teen", "20s", "30s", "40s", "60s"]
genders = ["man", "woman", "person"]
styles = ["realistic photo", "portrait photo", "high detail photo"]
lighting = ["soft studio light", "indoor light", "natural daylight"]
poses = ["frontal face", "slight head tilt", "looking at camera"]

def make_prompt(emotion):
    return (
        f"A {random.choice(genders)} in their {random.choice(ages)}, "
        f"{random.choice(poses)}, {random.choice(styles)}, "
        f"expressing {emotion} emotion, {random.choice(lighting)}"
    )

negative = "cartoon, anime, painting, extra limbs, deformed face, blurry, low quality, text, watermark"


In [5]:

raw_synth_root = Path("../data/synth_raw")  # temp
raw_synth_root.mkdir(parents=True, exist_ok=True)

def gen_images_for_class(emotion, n, seed0=1234):
    out_dir = raw_synth_root / emotion
    out_dir.mkdir(parents=True, exist_ok=True)

    g = torch.Generator(device=device).manual_seed(seed0)
    for i in tqdm(range(n), desc=f"Generating {emotion}"):
        prompt = make_prompt(emotion)
        img = pipe(
            prompt=prompt,
            negative_prompt=negative,
            num_inference_steps=30,
            guidance_scale=7.0,
            generator=g
        ).images[0]

        img.save(out_dir / f"{emotion}_synth_{seed0}_{i:05d}.png")

for c, n in need.items():
    if n > 0:
        gen_images_for_class(c, n)


Generating surprised: 100%|██████████| 1402/1402 [1:10:23<00:00,  3.01s/it]


In [9]:
# Preprocessing settings
OUT_SIZE = 128         # final resolution (ConvNeXt-friendly; also fine for ViT-S)
MARGIN = 0.30          # crop margin around face box
USE_CLAHE = True       # illumination normalization

In [12]:
def clamp_box(box, w, h):
    x1, y1, x2, y2 = box
    x1 = max(0, min(w - 1, x1))
    y1 = max(0, min(h - 1, y1))
    x2 = max(0, min(w - 1, x2))
    y2 = max(0, min(h - 1, y2))
    if x2 <= x1: x2 = min(w - 1, x1 + 1)
    if y2 <= y1: y2 = min(h - 1, y1 + 1)
    return np.array([x1, y1, x2, y2], dtype=np.float32)

def expand_box(box, w, h, margin=0.3):
    x1, y1, x2, y2 = box.astype(np.float32)
    bw = x2 - x1
    bh = y2 - y1
    x1 -= bw * margin
    y1 -= bh * margin
    x2 += bw * margin
    y2 += bh * margin
    return clamp_box([x1, y1, x2, y2], w, h)

def apply_clahe_gray(gray_u8):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray_u8)

def to_gray_u8(pil_img):
    arr = np.array(pil_img.convert("L"))
    if arr.dtype != np.uint8:
        arr = arr.astype(np.uint8)
    return arr

In [10]:
def align_by_eyes(gray, landmarks, out_size=128):
    """
    gray: HxW uint8 (cropped region)
    landmarks: (5,2) float in crop coords: left_eye, right_eye, nose, mouth_left, mouth_right
    """
    if landmarks is None or len(landmarks) != 5:
        return cv2.resize(gray, (out_size, out_size), interpolation=cv2.INTER_CUBIC)

    left_eye = landmarks[0]
    right_eye = landmarks[1]

    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    eyes_center = ((left_eye[0] + right_eye[0]) / 2.0, (left_eye[1] + right_eye[1]) / 2.0)
    h, w = gray.shape[:2]
    M = cv2.getRotationMatrix2D(eyes_center, angle, 1.0)

    rotated = cv2.warpAffine(
        gray, M, (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )

    # rotate landmarks too (for a stable crop)
    ones = np.ones((5, 1), dtype=np.float32)
    pts = np.hstack([landmarks.astype(np.float32), ones])  # 5x3
    rot_pts = (M @ pts.T).T  # 5x2

    nose = rot_pts[2]
    cx, cy = float(nose[0]), float(nose[1])

    eye_dist = float(np.linalg.norm(rot_pts[1] - rot_pts[0]))
    crop_half = max(out_size * 0.6, eye_dist * 1.8)  # heuristic

    x1 = int(cx - crop_half); y1 = int(cy - crop_half)
    x2 = int(cx + crop_half); y2 = int(cy + crop_half)

    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w, x2); y2 = min(h, y2)

    face = rotated[y1:y2, x1:x2]
    if face.size == 0:
        face = rotated

    return cv2.resize(face, (out_size, out_size), interpolation=cv2.INTER_CUBIC)


def preprocess_image(path, out_size=128, margin=0.3, use_clahe=True):
    """
    Returns a processed grayscale face image (uint8, out_size x out_size).
    Never drops the sample: falls back to simple resize if detection fails.
    """
    pil = Image.open(path).convert("RGB")
    w, h = pil.size
    gray = to_gray_u8(pil)

    boxes, probs, landmarks = mtcnn.detect(pil, landmarks=True)

    if boxes is None or len(boxes) == 0:
        face = cv2.resize(gray, (out_size, out_size), interpolation=cv2.INTER_CUBIC)
        detected = False
    else:
        best_i = int(np.argmax(probs))
        box = clamp_box(boxes[best_i], w, h)
        box = expand_box(box, w, h, margin=margin)

        x1, y1, x2, y2 = box.astype(int)
        crop_gray = gray[y1:y2, x1:x2]

        # shift landmarks into crop coordinates
        lm = landmarks[best_i].copy()
        lm[:, 0] -= x1
        lm[:, 1] -= y1

        face = align_by_eyes(crop_gray, lm, out_size=out_size)
        detected = True

    if use_clahe:
        face = apply_clahe_gray(face)

    return face, detected


In [22]:
processed_train_out = Path("../data/dataset_processed_split_synth/train")
processed_train_out.mkdir(parents=True, exist_ok=True)

# Copy REAL train first (so train contains real+synthetic)
import shutil
real_train = Path("../data/dataset_processed_split/train")

for c in classes:
    (processed_train_out / c).mkdir(parents=True, exist_ok=True)
    for p in (real_train / c).glob("*.png"):
        shutil.copy(p, processed_train_out / c / p.name)

# Now preprocess synthetic and add to the same train folders
for c in classes:
    synth_class_dir = raw_synth_root / c
    if not synth_class_dir.exists():
        continue
    out_dir = processed_train_out / c
    for img_path in tqdm(list(synth_class_dir.glob("*.png")), desc=f"Preprocess synth {c}"):
        face, detected = preprocess_image(img_path, out_size=OUT_SIZE, margin=MARGIN, use_clahe=USE_CLAHE)
        if not detected: 
            continue
        cv2.imwrite(str(out_dir / img_path.name), face)


Preprocess synth surprised: 100%|██████████| 1402/1402 [00:44<00:00, 31.69it/s]


In [23]:
# print image counts after adding synthetic
final_counts = {c: len(list((processed_train_out/c).glob("*.png"))) for c in classes}
final_counts

{'angry': 3736,
 'disgusted': 3619,
 'fearful': 3743,
 'happy': 5413,
 'neutral': 3784,
 'sad': 3779,
 'surprised': 3744}